# 技能5 · Day 7 上机：端到端Capstone整合 -- 营销策略Agent系统的因果评估

**版本**：v5.0 学习材料包（收官）
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **DSR六步框架** 规划Capstone，把技能1-5整合为完整系统
2. 跑通端到端流水线：**causaldata NSW** -> **DoWhy因果估计** -> **LangGraph Agent** -> **deepeval评估** -> **IMRaD论文草稿**
3. 写出含DSR artifact描述的论文草稿，理解天道推演×多Agent仿真的同构关系

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：causaldata（NSW真实RCT）+ DoWhy（因果推断）+ LangGraph（Agent编排）+ deepeval（评估）。
Capstone场景：为营销策略Agent系统做因果评估 -- 从数据到论文，端到端跑通。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ deepeval 的 GEval 默认使用 OpenAI 作为 judge 模型，需设置 `OPENAI_API_KEY`。
> 自定义 BaseMetric 不需要 API Key，可直接运行。
> DoWhy + LangGraph 也不需要 API Key，可直接运行。

In [ ]:
# !pip install causaldata dowhy langgraph deepeval -q
# export OPENAI_API_KEY=sk-...  # 仅GEval需要，自定义BaseMetric不需要

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# 因果推断（技能3）
from dowhy import CausalModel

# Agent编排（技能5 Day2）
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

# 评估框架（技能5 Day3）
from deepeval import evaluate
from deepeval.metrics import GEval, BaseMetric
from deepeval.test_case import LLMTestCase
try:
    from deepeval.test_case import LLMTestCaseParams
except ImportError:
    from deepeval.test_case import SingleTurnParams as LLMTestCaseParams

print("导入完成")
print("Capstone流水线：causaldata -> DoWhy -> LangGraph -> deepeval -> IMRaD")

## 1. Capstone概览：DSR框架 + 技能1-5整合

本Day是技能5的收官，也是整个课程的Capstone启动点。用一个真实数据集（NSW职业培训实验），串起因果分析->Agent构建->评估->论文的完整流水线。

### 端到端流水线
```
数据层(causaldata NSW真实RCT)
  ↓ 定义 treatment/outcome/covariates
因果层(DoWhy估计ATE)
  ↓ "营销干预的因果效果是多少？"
Agent层(LangGraph编排营销策略Agent)
  ↓ Agent调用因果分析结果->生成策略
评估层(deepeval + LLM-as-a-judge)
  ↓ 评估Agent策略质量
论文层(IMRaD草稿 + DSR artifact描述)
  ↓ 可复现的研究贡献
```

### 技能1-5映射
| 技能 | 在Capstone中的角色 | 真实库 |
|:----:|-------------------|--------|
| 技能1 | 表示工程（Agent知识基础） | embeddings |
| 技能2 | 原生架构（LangGraph图结构） | LangGraph |
| 技能3 | 因果推断（ATE估计） | DoWhy + causaldata |
| 技能4 | 商业模式（价值捕获） | 商业模式分析 |
| 技能5 | 系统工程（端到端构建+评估+论文） | deepeval + langsmith |

### DSR六步框架（Hevner 2004; Peffers 2007）
问题识别 -> 目标定义 -> 设计开发 -> 演示 -> 评估 -> 传播

你的Capstone就是一个DSR artifact -- 它的设计原则、评估方法、部署经验都是可发表的知识贡献。

In [ ]:
# TODO 1：DSR系统设计 -- 用DSR六步框架定义Capstone
# 提示：定义dsr_plan字典，六步各写1-2句描述
#   参考：Hevner et al. (2004) MIS Quarterly; Peffers et al. (2007) JMIS
# 要求：为"营销策略Agent系统的因果评估"Capstone定义DSR计划

# ===== 你的代码 =====
dsr_plan = {
    "problem_identification": "",   # 营销面临什么问题？
    "objectives": "",               # artifact应达到什么效果？
    "design_development": "",       # 架构/Agent/评估设计
    "demonstration": "",            # 在什么场景演示？
    "evaluation": "",               # 评估方法
    "communication": ""             # 如何传播
}
# ====================

print("DSR六步计划：")
for step, desc in dsr_plan.items():
    print(f"  [{step}] {desc[:60]}...")

## 2. 数据层：NSW真实RCT数据

**NSW职业培训实验**：因果推断领域的经典真实RCT数据集。1970年代在美国随机分配失业人员到职业培训组（treatment）和对照组（control），追踪后续收入变化。

**营销映射**：把NSW当作营销A/B测试数据 --
| NSW变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否收到个性化营销 | 处理 T |
| `re78` | 营销后转化率/GMV | 结果 Y |
| `re75` | 实验前历史消费 | CUPED协变量 |
| `age`,`education`,... | 用户画像 | 协变量 X |

这一步复用技能3的真实数据集，但今天把它作为Agent流水线的输入。

In [ ]:
# TODO 2：数据层 -- 加载真实NSW数据，定义treatment/outcome/covariates
# 提示：from causaldata import nsw; df = nsw.load_pandas().data
#   NSW变量营销映射：treat=营销干预, re78=转化率, re75=基线消费
# 要求：加载数据，定义变量名，打印形状和前5行

# ===== 你的代码 =====
from causaldata import nsw

df = None               # 加载NSW数据
TREATMENT = None        # 处理变量名（字符串）
OUTCOME = None          # 结果变量名（字符串）
COVARIATES = None       # 协变量列表
# ====================

print(f"数据形状: {df.shape}")
print(f"处理: {TREATMENT}, 结果: {OUTCOME}")
print(f"协变量: {COVARIATES}")
print(f"\n处理组 vs 对照组样本量:")
print(df[TREATMENT].value_counts())
df.head()

## 3. 因果层：用DoWhy估计ATE

**核心问题**：营销干预对转化率的因果效果（ATE）是多少？

用DoWhy的"假设->识别->估计->反驳"四步流程：
1. **假设**：构建因果图，指定treatment/outcome/common_causes
2. **识别**：DoWhy自动找后门调整集
3. **估计**：用backdoor.linear_regression估计ATE
4. **反驳**：安慰剂检验验证稳健性

这一步的ATE估计结果将作为Agent的输入 -- Agent读取因果证据来生成策略。

In [ ]:
# TODO 3：因果层 -- 用DoWhy估计ATE，回答"营销干预的因果效果"
# 提示：
#   model = CausalModel(data=df, treatment=TREATMENT, outcome=OUTCOME, common_causes=COVARIATES)
#   estimand = model.identify_effect(proceed_when_unidentifiable=True)
#   estimate = model.estimate_effect(estimand, method_name="backdoor.linear_regression")
#   print(estimate.value)  # ATE
# 要求：构建因果模型，识别估计，打印ATE值

# ===== 你的代码 =====
model = None        # 构建CausalModel
estimand = None     # 识别因果效应
estimate = None     # 估计ATE
# ====================

print(f"ATE（平均处理效应）: {estimate.value:.2f}")
print(f"解释: 营销干预使结果变量{'增加' if estimate.value > 0 else '减少'} {abs(estimate.value):.2f}")

## 4. Agent层：用LangGraph构建营销策略Agent

Agent架构（三个节点的StateGraph）：
```
START -> [analyze_causal] -> [generate_strategy] -> [review_strategy] -> END
         读取ATE，生成     基于因果证据      审核策略合规性
         因果分析摘要      生成营销策略
```

**关键设计**：Agent不是随机生成策略，而是**基于因果证据**生成策略 --
如果ATE正向显著，建议扩大投放；如果不显著，建议先做用户分群因果分析。

这是天道推演的核心思想：基于因果链和模式识别的逻辑推演，而非占卜。
Agent系统本质上是一个计算化的天道推演沙盘。

In [ ]:
# TODO 4：Agent层 -- 用LangGraph构建营销策略Agent
# 提示：
#   1. 定义AgentState(TypedDict) 包含 causal_ate, strategy, review_passed
#   2. 定义三个节点函数：analyze_causal / generate_strategy / review_strategy
#   3. 用StateGraph编排：START -> analyze -> generate -> review -> END
# 要求：构建可运行的LangGraph Agent，调用因果分析结果生成策略

# ===== 你的代码 =====
class AgentState(TypedDict):
    causal_ate: float
    strategy: str
    review_passed: bool

def analyze_causal(state: AgentState) -> dict:
    # 读取ATE，生成因果分析摘要
    pass

def generate_strategy(state: AgentState) -> dict:
    # 基于因果证据生成营销策略
    pass

def review_strategy(state: AgentState) -> dict:
    # 审核策略是否合规
    pass

# 构建并编译图
agent_app = None  # 构建StateGraph并compile
# ====================

# 运行Agent
result = agent_app.invoke({"causal_ate": estimate.value, "strategy": "", "review_passed": False})
print("Agent生成的策略：")
print(result["strategy"][:200])
print(f"\n审核通过: {result['review_passed']}")

## 5. 评估层：用deepeval评估Agent输出质量

评估Agent生成的营销策略质量，检查五个维度：

| 检查项 | 说明 | 为什么重要 |
|--------|------|------------|
| 有因果依据 | 策略是否引用ATE | 策略不能凭空生成 |
| 有预算建议 | 是否给出预算分配 | 营销策略必须可执行 |
| 有评估方案 | 是否含评估方法 | 可复现性要求 |
| 有目标人群 | 是否指定目标用户 | 个性化要求 |
| 无虚假承诺 | 没有"保证""100%" | 合规要求 |

用自定义 **BaseMetric**（不需要API Key）做规则检查，
进阶可用 **GEval**（LLM-as-a-judge，需OPENAI_API_KEY）做语义评估。

In [ ]:
# TODO 5：评估层 -- 用deepeval评估Agent输出质量
# 提示：
#   1. 创建LLMTestCase from Agent输出
#   2. 自定义BaseMetric评估策略质量（因果依据/预算建议/评估方案/合规性）
#   3. （可选）用GEval做LLM-as-a-judge自动评分（需OPENAI_API_KEY）
# 要求：定义StrategyQualityMetric，评估Agent策略质量

# ===== 你的代码 =====
class StrategyQualityMetric(BaseMetric):
    """评估营销策略质量：因果依据/预算/评估/合规"""
    def __init__(self, threshold=0.7):
        self.threshold = threshold
    # 实现 measure / a_measure / is_successful / __name__
    pass

# 创建测试用例
test_case = None  # 用LLMTestCase包装Agent输出
# 评估
quality_metric = None  # StrategyQualityMetric实例
# ====================

quality_metric.measure(test_case)
print(f"策略质量评分: {quality_metric.score:.2f}")
print(f"理由: {quality_metric.reason}")

## 6. 论文草稿：用IMRaD+DSR artifact描述

把Capstone整合为一篇IMRaD结构论文草稿：

| 章节 | 内容 | DSR映射 |
|------|------|---------|
| Introduction | 研究问题+贡献 | DSR Step 1: 问题识别 |
| Methods | 系统架构+评估方法 | DSR Step 3: 设计开发 |
| Results | ATE+Agent评估 | DSR Step 5: 评估 |
| Discussion | 发现+局限+未来 | DSR Step 6: 传播 |

整合前面所有步骤的产出：DSR计划(Step1) + ATE(Step3) + Agent结果(Step4) + 评估分数(Step5)。

In [ ]:
# TODO 6：论文草稿 -- 用IMRaD结构+DSR artifact描述
# 提示：
#   1. 定义generate_paper_draft函数，整合dsr_plan/ate/agent_result/eval_score
#   2. 用IMRaD结构：Introduction/Methods/Results/Discussion
#   3. Methods中包含DSR artifact描述
# 要求：生成结构化论文草稿（每节3-5句）

# ===== 你的代码 =====
def generate_paper_draft(dsr_plan, ate_value, agent_result, eval_score):
    """生成IMRaD论文草稿"""
    draft = {
        "title": "",        # 论文标题
        "abstract": "",     # 250字以内摘要
        "introduction": "", # 研究问题+贡献
        "methods": "",      # DSR框架+系统架构+评估方法
        "results": "",      # ATE+Agent评估结果
        "discussion": ""    # 发现+局限+未来方向
    }
    return draft

paper = None  # 调用generate_paper_draft
# ====================

for section, content in paper.items():
    print(f"\n{'='*50}")
    print(f"  {section.upper()}")
    print(f"{'='*50}")
    print(content)

## 7. 反思与前沿

### 反思问题
1. 你的Capstone在DSR六步中哪一步最薄弱？如何改进？
2. Agent的策略是否真正基于因果证据？如果ATE为负，Agent会怎么建议？
3. 自定义BaseMetric（规则检查）vs GEval（LLM-as-a-judge），各自的优劣势？
4. 如果要把这个Capstone投稿到Decision Support Systems，还需要补什么？

### 2026前沿：DSR + 可复现研究 + 天道推演×多Agent仿真

**DSR（设计科学研究）**：Hevner et al. (2004) MIS Quarterly的经典框架，在AI原生系统时代获得新生命。Agent系统本身是artifact，其架构模式、评估框架、安全实践都是可发表的DSR知识贡献。

**可复现研究**：开源代码+测试套件(deepeval CI)+trace存档(langsmith)+数据文档，让他人能独立复现结果。

**天道推演×多Agent仿真**（特色章节）：天道推演的沙盘模拟（因果链追踪+多路径概率评估）与多Agent仿真（Agent交互+涌现行为预测）共享同一因果建模底层。你的营销Agent系统本质上是一个计算化的天道推演沙盘-- Agent在模拟不同营销策略的因果效果，选择最优路径。这为工程系统提供了哲学层面的理论锚点。

参考 [Hevner 2004](https://www.jstor.org/stable/25148625) + [Peffers 2007](https://desrist.org/desrist/files/peffers2007.pdf) + [DoWhy](https://github.com/py-why/dowhy) + [LangGraph](https://github.com/langchain-ai/langgraph) + [deepeval](https://github.com/confident-ai/deepeval)。